In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from PIL import Image
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm  # Import tqdm for progress bars

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom Loss Function: Cosine/Sine Loss
class CosineSineLoss(nn.Module):
    def __init__(self):
        super(CosineSineLoss, self).__init__()

    def forward(self, preds, targets):
        cos_pred, sin_pred = preds[:, 0], preds[:, 1]
        cos_target = torch.cos(targets * torch.pi / 180)
        sin_target = torch.sin(targets * torch.pi / 180)
        loss = torch.mean((cos_pred - cos_target) ** 2 + (sin_pred - sin_target) ** 2)
        return loss

# Mean Angular Error for evaluation
class MeanAngularErrorLoss(nn.Module):
    def __init__(self):
        super(MeanAngularErrorLoss, self).__init__()

    def forward(self, preds, targets):
        preds = preds % 360
        targets = targets % 360
        diff = torch.abs(preds - targets)
        angular_error = torch.min(diff, 360 - diff)
        return torch.mean(angular_error)

# Dataset for Training and Validation
class AngleImageDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None, is_train=False):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        self.is_train = is_train

        # Filter angles < 360 for training data only
        if self.is_train:
            self.data = self.data[self.data['angle'].astype(float) < 360]
            print(f"Training dataset size after filtering angles < 360: {len(self.data)}")

        # Check validation dataset size (must be 369)
        if 'val' in csv_file and len(self.data) != 369:
            raise ValueError(f"Validation dataset must have exactly 369 samples, got {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        filename = row['filename']
        angle = float(row['angle']) % 360

        img_path = os.path.join(self.image_dir, filename)
        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(angle, dtype=torch.float32), filename

# Dataset for Test (No CSV, only images)
class TestImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.filenames = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png','.JPEG'))]
        if len(self.filenames) != 369:
            raise ValueError(f"Test dataset must have exactly 369 samples, got {len(self.filenames)}")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.image_dir, filename)
        if not os.path.isfile(img_path):
            raise FileNotFoundError(f"Image not found: {img_path}")

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, filename

# Convert predictions to angles
def preds_to_angle(preds):
    cos_pred, sin_pred = preds[:, 0], preds[:, 1]
    angle = torch.atan2(sin_pred, cos_pred) * 180 / torch.pi
    return angle % 360

# Data Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test transform (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# File paths

CSV_FILE = "/kaggle/input/data-campus/data/labels_train.csv"
IMAGE_DIR = "/kaggle/input/data-campus/data/images_train"
VAL_CSV_FILE = "/kaggle/input/data-campus/data/labels_val.csv"
VAL_IMAGE_DIR = "/kaggle/input/data-campus/data/images_val"
TEST_DIR = "/kaggle/input/data-campus/images_test/images_test"

# Load Datasets
train_dataset = AngleImageDataset(CSV_FILE, IMAGE_DIR, transform=transform, is_train=True)
val_dataset = AngleImageDataset(VAL_CSV_FILE, VAL_IMAGE_DIR, transform=test_transform, is_train=False)
test_dataset = TestImageDataset(TEST_DIR, transform=test_transform)

# DataLoaders
dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# Model
class AngleRegressionModel(nn.Module):
    def __init__(self):
        super(AngleRegressionModel, self).__init__()
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        # Remove the default classifier
        self.backbone.classifier = nn.Identity()
        # Add custom classifier with proper feature reduction
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # Reduce spatial dimensions to 1x1
            nn.Flatten(),             # Flatten to (batch_size, 768)
            nn.Linear(768, 512),      # 768 is the number of channels from convnext_tiny
            nn.ReLU(),
            nn.Linear(512, 2)         # Output [cos(angle), sin(angle)]
        )

    def forward(self, x):
        x = self.backbone.features(x)  # Extract features
        x = self.classifier(x)         # Apply custom classifier
        return x

# Model, Loss, Optimizer
model = AngleRegressionModel().to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model = model.to(device)

loss_fn = CosineSineLoss()
mae_fn = MeanAngularErrorLoss()  # For evaluation
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=50)

# Training Loop
best_val_loss = float('inf')
patience = 5
counter = 0
num_epochs = 30

for epoch in range(num_epochs):
    # Training
    model.train()
    total_loss = 0
    # Wrap dataloader with tqdm for progress bar
    with tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False) as pbar:
        for imgs, angles, _ in pbar:
            imgs, angles = imgs.to(device), angles.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, angles)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            # Update progress bar with current loss
            pbar.set_postfix({'loss': loss.item()})
    avg_train_loss = total_loss / len(dataloader.dataset)

    # Validation
    model.eval()
    total_val_loss = 0
    preds_all, targets_all, val_filenames = [], [], []
    # Wrap val_loader with tqdm for progress bar
    with torch.no_grad(), tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False) as pbar:
        for imgs, angles, filenames in pbar:
            imgs, angles = imgs.to(device), angles.to(device)
            preds = model(imgs)
            val_loss = loss_fn(preds, angles)
            total_val_loss += val_loss.item() * imgs.size(0)
            pred_angles = preds_to_angle(preds)
            preds_all.extend(pred_angles.cpu().numpy())
            targets_all.extend(angles.cpu().numpy())
            val_filenames.extend(filenames)
            # Update progress bar with current validation loss
            pbar.set_postfix({'val_loss': val_loss.item()})

    avg_val_loss = total_val_loss / len(val_loader.dataset)
    mae = mae_fn(torch.tensor(preds_all), torch.tensor(targets_all)).item()

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, MAE: {mae:.4f}")

    # Save validation predictions to CSV (only for best epoch)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        # Save the state_dict of the model (not the DataParallel wrapper)
        torch.save(model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict(), 'best_model.pth')
        counter = 0

        # Create validation predictions DataFrame with id (0–368)
        val_predictions_df = pd.DataFrame({
            'id': range(369),
            'angle': preds_all
        })
        val_predictions_df.to_csv('predictions.csv', mode='w', index=False)
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

    scheduler.step()

# Load best model for test predictions
model = AngleRegressionModel().to(device)
model.load_state_dict(torch.load('best_model.pth'))
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)
model.eval()

# Test Predictions
test_filenames, test_pred_angles = [], []
with torch.no_grad():
    # Wrap test_loader with tqdm for progress bar
    with tqdm(test_loader, desc="Test", leave=False) as pbar:
        for imgs, filenames in pbar:
            imgs = imgs.to(device)
            preds = model(imgs)
            pred_angles = preds_to_angle(preds)
            test_filenames.extend(filenames)
            test_pred_angles.extend(pred_angles.cpu().numpy())

# Save test predictions to CSV (append with id 369–737)
test_predictions_df = pd.DataFrame({
    'id': range(369, 738),
    'angle': test_pred_angles
})
test_predictions_df.to_csv('predictions.csv', mode='a', header=False, index=False)

# Verify CSV row count
predictions_df = pd.read_csv('predictions.csv')
print(f"Total rows in predictions.csv (including header): {len(predictions_df) + 1}")
print("Predictions saved to predictions.csv")

Training dataset size after filtering angles < 360: 6537


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:00<00:00, 192MB/s] 


Using 2 GPUs!


Epoch 1/30, Train Loss: 0.8762, Val Loss: 0.7378, MAE: 52.9712


Epoch 2/30, Train Loss: 0.6336, Val Loss: 0.5623, MAE: 42.9467


...

Epoch 27/30, Train Loss: 0.0209, Val Loss: 0.3364, MAE: 26.3301


Epoch 28/30, Train Loss: 0.0209, Val Loss: 0.3183, MAE: 25.0183


Epoch 29/30, Train Loss: 0.0178, Val Loss: 0.3239, MAE: 25.4818


Epoch 30/30, Train Loss: 0.0164, Val Loss: 0.3234, MAE: 25.5595


Total rows in predictions.csv (including header): 739
Predictions saved to predictions.csv
